# HPLC Substrate Analysis Pipeline — Demo

**목적**: HPLC 시계열 CSV를 넣으면 기질 소비 kinetics 파라미터를 자동으로 추출하는 파이프라인 시연

이 노트북은 3가지를 보여줍니다:
1. **입력 데이터** — 어떤 형태의 CSV가 들어가는지
2. **계산 방법** — 각 파라미터가 어떤 수식으로 계산되는지
3. **출력 결과** — 시각화와 요약 테이블

---

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.makedirs('../output', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

from hplc_analysis.pipeline import analyze_experiment, results_to_summary_df

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# ── 데이터 선택 ──────────────────────────────────────────────
# 아래 DATASET 값을 바꿔서 분석 대상을 전환하세요.
#   '1:1'  →  Glucose 15 g/L : Xylose 16 g/L
#   '2:1'  →  Glucose 20 g/L : Xylose 10 g/L

DATASET = '1:1'   # ◀◀◀  여기를 '2:1' 로 바꾸면 2:1 데이터 분석

files = {
    '1:1': '../data/1-to-1-GlucoseXyloseMicroplateGrowth (1).csv',
    '2:1': '../data/2-to-1-GlucoseXyloseMicroplateGrowth (1).csv',
}
filepath = files[DATASET]
print(f'Selected dataset: {DATASET}  →  {filepath}')

---
## 1. Input: 이런 데이터가 들어갑니다

HPLC로 측정한 기질 농도 시계열 CSV.  
각 시간대마다 **3~9개 replicate** 측정값이 있습니다.

In [ ]:
raw = pd.read_csv(filepath, encoding='utf-8-sig')
raw.columns = [c.strip() for c in raw.columns]

print(f'CSV shape: {raw.shape[0]} rows x {raw.shape[1]} columns')
print(f'Columns:   {list(raw.columns)}')
print()
raw.head(10)

In [ ]:
# Replicate 수 확인
rep_counts = raw.groupby('Time (h)').size()
print('Time (h)  →  Replicates')
print(rep_counts.to_string())
print(f'\nTotal timepoints: {len(rep_counts)}')
print(f'Time range: {raw["Time (h)"].min():.1f} ~ {raw["Time (h)"].max():.1f} h')

In [ ]:
# 원본 데이터 시각화 (Replicate 별 scatter)
fig, ax = plt.subplots(figsize=(11, 5))

ax.scatter(raw['Time (h)'], raw['[Glucose] (g/L)'],
           c='#2196F3', alpha=0.6, s=30, label='Glucose (each replicate)', edgecolors='white', linewidth=0.3)
ax.scatter(raw['Time (h)'], raw['[Xylose] (g/L)'],
           c='#FF9800', alpha=0.6, s=30, label='Xylose (each replicate)', edgecolors='white', linewidth=0.3)

ax.set_xlabel('Time (h)')
ax.set_ylabel('Concentration (g/L)')
ax.set_title(f'Raw Input Data — {DATASET} Glucose:Xylose', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('../output/demo_1_raw_input.png', dpi=150, bbox_inches='tight')
plt.show()
print('↑ 이 데이터를 파이프라인에 넣습니다.')

---
## 2. Pipeline 실행: 한 줄로 전체 분석

```python
results = analyze_experiment('path/to/data.csv')
```

내부적으로 **4단계**가 순차 실행됩니다:

| 단계 | 모듈 | 역할 |
|:---:|:---:|:---|
| 1 | `loader.py` | CSV 로딩, 컬럼 자동감지, 검증 |
| 2 | `stats.py` | Replicate 집계 (mean, SD, CI), outlier flagging |
| 3 | `kinetics.py` | 소비율, lag phase, depletion time 계산 |
| 4 | `diauxic.py` | 교차 기질 diauxic shift 분석 |

In [ ]:
results = analyze_experiment(filepath)
print(f'\nAnalysis complete for: {DATASET}')
print(f'  Substrates found: {list(results.kinetics.keys())}')
print(f'  Diauxic detected: {results.diauxic.diauxic_detected}')

---
## 3. 계산 방법: 각 파라미터는 이렇게 계산됩니다

### 3-A. Replicate 통계 (stats.py)

각 시간대 replicate 측정값을 집계합니다:

| 통계량 | 수식 |
|:---:|:---|
| Mean | $\bar{S} = \frac{1}{n} \sum S_i$ |
| SD | $\sigma = \sqrt{\frac{\sum(S_i - \bar{S})^2}{n-1}}$ |
| SEM | $\text{SEM} = \sigma / \sqrt{n}$ |
| 95% CI | $\bar{S} \pm t_{0.975, n-1} \times \text{SEM}$ |
| CV (%) | $\text{CV} = (\sigma / \bar{S}) \times 100$ |

In [ ]:
# Replicate 통계 테이블 보기
for substrate in results.stats:
    print(f'\n=== {substrate} — Replicate Statistics ===')
    st = results.stats[substrate].copy()
    st = st.round(3)
    print(st.to_string())

In [ ]:
# Mean ± 95% CI 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Glucose': '#2196F3', 'Xylose': '#FF9800'}

for ax, substrate in zip(axes, results.stats):
    st = results.stats[substrate]
    col = f'{substrate} (g/L)'
    c = colors[substrate]

    # CI band
    ax.fill_between(st.index, st['ci95_lower'], st['ci95_upper'],
                    alpha=0.25, color=c, label='95% CI')
    # Mean
    ax.plot(st.index, st['mean'], 'o-', color=c, linewidth=2, markersize=5, label='Mean')
    # Replicates
    ax.scatter(results.raw_df['Time (h)'], results.raw_df[col],
               color=c, alpha=0.3, s=12, zorder=1, label='Replicates')

    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Concentration (g/L)')
    ax.set_title(f'{substrate} — Mean ± 95% CI', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'{DATASET} Experiment — Replicate Aggregation', fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('../output/demo_2_replicate_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('↑ Replicate scatter → Mean ± 95% CI band 로 집계')

### 3-B. 소비율 계산 (kinetics.py)

시간 간격이 **비균일**하므로 (1.7h ~ 25.7h), 일반 `np.diff`가 아닌 **non-uniform central finite difference**를 사용합니다:

$$
\frac{dS}{dt}\bigg|_i = 
\frac{-h_2}{h_1(h_1+h_2)} S_{i-1}
+ \frac{h_2-h_1}{h_1 h_2} S_i
+ \frac{h_1}{h_2(h_1+h_2)} S_{i+1}
$$

여기서 $h_1 = t_i - t_{i-1}$, $h_2 = t_{i+1} - t_i$

**Consumption rate** = $-dS/dt$ (양수면 기질이 소비되는 중)

In [ ]:
# 시간 간격이 얼마나 비균일한지 확인
for substrate in results.kinetics:
    kin = results.kinetics[substrate]
    intervals = np.diff(kin.times)
    print(f'{substrate} — Time intervals (h): {np.round(intervals, 2)}')
    print(f'  Min: {intervals.min():.2f}h, Max: {intervals.max():.2f}h, '
          f'Ratio: {intervals.max()/intervals.min():.1f}x')
    print()

In [ ]:
# 소비율 시각화 + q_max 표시
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, substrate in zip(axes, results.kinetics):
    kin = results.kinetics[substrate]
    c = colors[substrate]

    ax.plot(kin.times, kin.rates, 'o-', color=c, linewidth=2, markersize=6)
    ax.plot(kin.t_qmax, kin.q_max, '*', color='red', markersize=18, zorder=5)
    ax.annotate(f'q_max = {kin.q_max:.3f} g/L/h\nt = {kin.t_qmax:.1f}h',
                xy=(kin.t_qmax, kin.q_max), xytext=(kin.t_qmax+5, kin.q_max*0.85),
                fontsize=10, fontweight='bold', color='red',
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.fill_between(kin.times, 0, kin.rates, where=(kin.rates > 0), alpha=0.1, color=c)

    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Consumption Rate (g/L/h)')
    ax.set_title(f'{substrate} — rate = $-dS/dt$', fontweight='bold')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'{DATASET} Experiment — Non-uniform Finite Difference Rates',
             fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('../output/demo_3_consumption_rates.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-C. Lag Phase — Tangent-Line Intersection Method

$q_{max}$ 지점에서의 접선이 $S = S_0$ 수평선과 만나는 점 → **lag time**

$$
\text{Tangent: } S(t) = S_{q_{max}} + \left(\frac{dS}{dt}\bigg|_{q_{max}}\right) \cdot (t - t_{q_{max}})
$$

$$
t_{lag} = t_{q_{max}} + \frac{S_0 - S_{q_{max}}}{dS/dt|_{q_{max}}}
$$

In [ ]:
# Lag phase tangent-line 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, substrate in zip(axes, results.kinetics):
    kin = results.kinetics[substrate]
    c = colors[substrate]

    # Mean concentration curve
    ax.plot(kin.times, kin.means, 'o-', color=c, linewidth=2, markersize=5, label='Mean conc.')

    # S0 horizontal line
    ax.axhline(kin.S0, color='green', linestyle=':', alpha=0.7, linewidth=1.5, label=f'$S_0$ = {kin.S0:.1f}')

    # Tangent line at q_max point
    slope = -kin.q_max  # dS/dt is negative
    t_tangent = np.linspace(kin.t_lag - 3, kin.t_qmax + 8, 100)
    S_qmax = kin.means[np.argmin(np.abs(kin.times - kin.t_qmax))]
    S_tangent = S_qmax + slope * (t_tangent - kin.t_qmax)
    ax.plot(t_tangent, S_tangent, '--', color='red', linewidth=2, alpha=0.8, label='Tangent at $q_{max}$')

    # t_lag marker
    ax.axvline(kin.t_lag, color='purple', linestyle='-.', alpha=0.7, linewidth=1.5)
    ax.plot(kin.t_lag, kin.S0, 'D', color='purple', markersize=10, zorder=5)
    ax.annotate(f'$t_{{lag}}$ = {kin.t_lag:.1f}h',
                xy=(kin.t_lag, kin.S0), xytext=(kin.t_lag - 15, kin.S0 * 0.7),
                fontsize=11, fontweight='bold', color='purple',
                arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))

    # q_max point
    ax.plot(kin.t_qmax, S_qmax, '*', color='red', markersize=15, zorder=5)

    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Concentration (g/L)')
    ax.set_title(f'{substrate} — Lag Phase Detection', fontweight='bold')
    ax.legend(fontsize=9, loc='lower left')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=-1)

fig.suptitle(f'{DATASET} Experiment — Tangent-Line Intersection Method',
             fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('../output/demo_4_lag_phase.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-D. Depletion / t₅₀ / t₉₀ — Linear Interpolation

농도가 threshold를 지나는 두 시점 사이에서 **선형 보간**:

$$
t_{target} = t_i + \frac{S_i - S_{target}}{S_i - S_{i+1}} \cdot (t_{i+1} - t_i)
$$

| 파라미터 | Target 농도 |
|:---:|:---|
| $t_{50}$ | $S_0 - 0.5 \cdot \Delta S$ |
| $t_{90}$ | $S_0 - 0.9 \cdot \Delta S$ |
| $t_{depletion}$ | $\max(0.5,\; 0.05 \times S_0)$ |

In [ ]:
# Depletion & milestone 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, substrate in zip(axes, results.kinetics):
    kin = results.kinetics[substrate]
    c = colors[substrate]

    ax.plot(kin.times, kin.means, 'o-', color=c, linewidth=2.5, markersize=5, label='Mean conc.', zorder=3)

    # 50% level
    S_50 = kin.S0 - 0.5 * kin.delta_S
    ax.axhline(S_50, color='#9C27B0', linestyle=':', alpha=0.5, linewidth=1)
    if kin.t_50:
        ax.plot(kin.t_50, S_50, 's', color='#9C27B0', markersize=10, zorder=5)
        ax.annotate(f'$t_{{50}}$ = {kin.t_50:.1f}h', xy=(kin.t_50, S_50),
                    xytext=(kin.t_50 + 4, S_50 + 1.5), fontsize=10, color='#9C27B0', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#9C27B0'))

    # 90% level
    S_90 = kin.S0 - 0.9 * kin.delta_S
    ax.axhline(S_90, color='#E91E63', linestyle=':', alpha=0.5, linewidth=1)
    if kin.t_90:
        ax.plot(kin.t_90, S_90, 's', color='#E91E63', markersize=10, zorder=5)
        ax.annotate(f'$t_{{90}}$ = {kin.t_90:.1f}h', xy=(kin.t_90, S_90),
                    xytext=(kin.t_90 + 4, S_90 + 1.5), fontsize=10, color='#E91E63', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#E91E63'))

    # Depletion
    dep_thresh = max(0.5, 0.05 * kin.S0)
    ax.axhline(dep_thresh, color='red', linestyle='--', alpha=0.4, linewidth=1)
    if kin.t_depletion:
        ax.axvline(kin.t_depletion, color='red', linestyle='--', alpha=0.5)
        ax.plot(kin.t_depletion, dep_thresh, 'v', color='red', markersize=12, zorder=5)
        ax.annotate(f'$t_{{dep}}$ = {kin.t_depletion:.1f}h', xy=(kin.t_depletion, dep_thresh),
                    xytext=(kin.t_depletion + 3, dep_thresh + 2.5), fontsize=10, color='red', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='red'))

    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Concentration (g/L)')
    ax.set_title(f'{substrate} — Milestones', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=-0.5)

fig.suptitle(f'{DATASET} Experiment — $t_{{50}}$, $t_{{90}}$, $t_{{depletion}}$ (Linear Interpolation)',
             fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('../output/demo_5_depletion_milestones.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-E. Diauxic Shift 분석 (diauxic.py)

두 기질의 **active consumption window** (rate > 10% of q_max)를 비교하여:

| 파라미터 | 의미 |
|:---:|:---|
| **primary_substrate** | 먼저 고갈되는 기질 |
| **diauxic_lag** | 1차 기질 고갈 → 2차 기질 본격 소비 시작 사이 시간 |
| **overlap_fraction** | 두 기질 동시 소비 시간 비율 |

In [ ]:
# Diauxic analysis 시각화
fig, ax = plt.subplots(figsize=(12, 6))
d = results.diauxic

for substrate, kin in results.kinetics.items():
    c = colors[substrate]
    ax.plot(kin.times, kin.means, 'o-', color=c, linewidth=2.5, markersize=5, label=substrate, zorder=3)

    # Active window bar
    if substrate in d.active_windows:
        t_s, t_e = d.active_windows[substrate]
        ax.axvspan(t_s, t_e, alpha=0.1, color=c)
        y_pos = kin.S0 * 1.05 if substrate == 'Glucose' else kin.S0 * 1.12
        ax.annotate('', xy=(t_s, y_pos), xytext=(t_e, y_pos),
                    arrowprops=dict(arrowstyle='<->', color=c, lw=2))
        ax.text((t_s + t_e) / 2, y_pos + 0.3, f'{substrate} active\n({t_s:.0f}~{t_e:.0f}h)',
                ha='center', fontsize=9, color=c, fontweight='bold')

    # Depletion line
    if kin.t_depletion:
        ax.axvline(kin.t_depletion, color=c, linestyle='--', alpha=0.5)

# Status box
box_text = (
    f"Diauxic shift: {'DETECTED' if d.diauxic_detected else 'Not detected'}\n"
    f"Primary: {d.primary_substrate}\n"
    f"Order: {' → '.join(d.consumption_order)}\n"
    f"Overlap: {d.overlap_fraction:.1%}"
)
ax.text(0.98, 0.98, box_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray', alpha=0.9))

ax.set_xlabel('Time (h)')
ax.set_ylabel('Concentration (g/L)')
ax.set_title(f'{DATASET} Experiment — Diauxic Shift Analysis', fontweight='bold')
ax.legend(loc='center left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('../output/demo_6_diauxic.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Output: 최종 결과 요약

### 4-A. 파라미터 요약 테이블

In [ ]:
# 파라미터 요약 테이블
rows = []
for substrate, kin in results.kinetics.items():
    rows.append({
        'Substrate': substrate,
        'S0 (g/L)': f'{kin.S0:.2f}',
        'Se (g/L)': f'{kin.Se:.3f}',
        'delta_S (g/L)': f'{kin.delta_S:.2f}',
        'Efficiency (%)': f'{kin.efficiency:.1f}',
        'q_avg (g/L/h)': f'{kin.q_avg:.3f}',
        'q_max (g/L/h)': f'{kin.q_max:.3f}',
        't_qmax (h)': f'{kin.t_qmax:.1f}',
        't_lag (h)': f'{kin.t_lag:.1f}',
        't_50 (h)': f'{kin.t_50:.1f}' if kin.t_50 else 'N/A',
        't_90 (h)': f'{kin.t_90:.1f}' if kin.t_90 else 'N/A',
        't_depletion (h)': f'{kin.t_depletion:.1f}' if kin.t_depletion else 'N/A',
        't_active (h)': f'{kin.t_active:.1f}' if kin.t_active else 'N/A',
        'q_active (g/L/h)': f'{kin.q_active:.3f}' if kin.q_active else 'N/A',
    })

summary_table = pd.DataFrame(rows).set_index('Substrate').T
summary_table.index.name = 'Parameter'
print(f'=== {DATASET} Experiment — All Kinetic Parameters ===')
print()
print(summary_table.to_string())

### 4-B. 종합 대시보드 (4-panel)

In [ ]:
# 종합 대시보드
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle(f'{DATASET} Glucose:Xylose — Full Analysis Dashboard',
             fontsize=15, fontweight='bold', y=1.01)

# ── Panel 1: Consumption curves with milestones ──
ax = axes[0, 0]
for substrate, kin in results.kinetics.items():
    c = colors[substrate]
    st = results.stats[substrate]
    ax.fill_between(st.index, st['ci95_lower'], st['ci95_upper'], alpha=0.2, color=c)
    ax.plot(kin.times, kin.means, 'o-', color=c, linewidth=2, markersize=4, label=substrate)
    if kin.t_depletion:
        ax.axvline(kin.t_depletion, color=c, linestyle='--', alpha=0.5)
ax.set_xlabel('Time (h)'); ax.set_ylabel('Conc. (g/L)')
ax.set_title('(A) Substrate Consumption', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Panel 2: Consumption rates ──
ax = axes[0, 1]
for substrate, kin in results.kinetics.items():
    c = colors[substrate]
    ax.plot(kin.times, kin.rates, 'o-', color=c, linewidth=2, markersize=4, label=substrate)
    ax.plot(kin.t_qmax, kin.q_max, '*', color='red', markersize=14, zorder=5)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Time (h)'); ax.set_ylabel('Rate (g/L/h)')
ax.set_title('(B) Consumption Rate ($-dS/dt$)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Panel 3: Normalized ──
ax = axes[0, 2]
for substrate, kin in results.kinetics.items():
    c = colors[substrate]
    norm = np.clip((kin.S0 - kin.means) / kin.delta_S, 0, 1.05) if kin.delta_S > 0 else np.zeros_like(kin.means)
    ax.plot(kin.times, norm, 'o-', color=c, linewidth=2, markersize=4, label=substrate)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='50%')
ax.axhline(0.9, color='gray', linestyle='--', alpha=0.5, label='90%')
ax.set_xlabel('Time (h)'); ax.set_ylabel('Fraction consumed')
ax.set_title('(C) Normalized Consumption', fontweight='bold')
ax.set_ylim(-0.05, 1.1); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Panel 4: CV% ──
ax = axes[1, 0]
for substrate, st in results.stats.items():
    c = colors[substrate]
    ax.plot(st.index, st['cv_pct'], 'o-', color=c, linewidth=2, markersize=4, label=substrate)
ax.axhline(20, color='red', linestyle='--', alpha=0.5, label='20% threshold')
ax.set_xlabel('Time (h)'); ax.set_ylabel('CV (%)')
ax.set_title('(D) Replicate Variability', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Panel 5: Diauxic summary ──
ax = axes[1, 1]
for substrate, kin in results.kinetics.items():
    c = colors[substrate]
    ax.plot(kin.times, kin.means, 'o-', color=c, linewidth=2, markersize=4, label=substrate)
    if substrate in d.active_windows:
        t_s, t_e = d.active_windows[substrate]
        ax.axvspan(t_s, t_e, alpha=0.08, color=c)
status = 'DETECTED' if d.diauxic_detected else 'Not detected'
ax.text(0.03, 0.97, f'Diauxic: {status}\nOrder: {" → ".join(d.consumption_order)}',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
ax.set_xlabel('Time (h)'); ax.set_ylabel('Conc. (g/L)')
ax.set_title('(E) Diauxic Analysis', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Panel 6: Parameter bar chart ──
ax = axes[1, 2]
substrates = list(results.kinetics.keys())
x = np.arange(5)
width = 0.35
for i, substrate in enumerate(substrates):
    kin = results.kinetics[substrate]
    vals = [kin.q_max, kin.q_avg, kin.q_active or 0,
            (kin.t_depletion or 0) / 10, (kin.t_active or 0) / 10]
    c = colors[substrate]
    bars = ax.bar(x + i * width - width/2, vals, width, color=c, alpha=0.7,
                  edgecolor='black', linewidth=0.5, label=substrate)
ax.set_xticks(x)
ax.set_xticklabels(['$q_{max}$\n(g/L/h)', '$q_{avg}$\n(g/L/h)', '$q_{active}$\n(g/L/h)',
                     '$t_{dep}$\n(/10, h)', '$t_{active}$\n(/10, h)'], fontsize=9)
ax.set_title('(F) Key Parameters Comparison', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

fig.tight_layout()
plt.savefig('../output/demo_7_full_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-C. CSV 출력

파이프라인은 두 종류의 CSV를 생성합니다:
- **summary.csv** — 실험 당 1행, 모든 파라미터 포함
- **timeseries.csv** — 시간대별 상세 데이터 (mean, SD, CI, rate, normalized consumption)

In [ ]:
from hplc_analysis.pipeline import export_results, results_to_timeseries_df

s_path, t_path = export_results(results, '../output', f'demo_{DATASET}')
print(f'Summary CSV:    {s_path}')
print(f'Timeseries CSV: {t_path}')

print('\n=== Summary CSV (transposed) ===')
summary_df = results_to_summary_df(results, DATASET)
print(summary_df.T.to_string())

print('\n=== Timeseries CSV (first 10 rows) ===')
ts_df = results_to_timeseries_df(results, DATASET)
print(ts_df.head(10).to_string(index=False))

---
## 5. 수식 요약표

| Parameter | Formula | Description |
|:---|:---|:---|
| $S_0$ | First timepoint mean | Initial concentration |
| $S_e$ | Last timepoint mean | Final concentration |
| $\Delta S$ | $S_0 - S_e$ | Total consumed |
| Efficiency | $\Delta S / S_0 \times 100$ | % consumed |
| $q_{avg}$ | $\Delta S / \Delta t$ | Average rate |
| $q_{max}$ | $\max(-dS/dt)$ via finite diff. | Peak rate |
| $t_{lag}$ | Tangent intersection with $S_0$ | Lag phase end |
| $t_{dep}$ | Interpolation at threshold | Depletion time |
| $t_{50}, t_{90}$ | Interpolation at 50%, 90% $\Delta S$ | Milestone times |
| $t_{active}$ | $t_{dep} - t_{lag}$ | Active consumption period |
| $q_{active}$ | $\Delta S / t_{active}$ | Active-phase rate |
| Diauxic lag | $t_{2nd\_active\_start} - t_{1st\_depletion}$ | Gap between substrates |

In [ ]:
# 수식 시각화 요약 figure
fig, ax = plt.subplots(figsize=(14, 7))

# Use Glucose as example
kin = results.kinetics['Glucose']
st = results.stats['Glucose']
c = '#2196F3'

# Main curve
ax.fill_between(st.index, st['ci95_lower'], st['ci95_upper'], alpha=0.15, color=c)
ax.plot(kin.times, kin.means, 'o-', color=c, linewidth=2.5, markersize=6, zorder=3)

# S0 line
ax.axhline(kin.S0, color='green', linestyle=':', alpha=0.6, linewidth=1.5)
ax.text(kin.times[-1] + 1, kin.S0, f'$S_0$ = {kin.S0:.1f}', fontsize=11, color='green', va='center')

# Tangent line at q_max
slope_val = -kin.q_max
i_qmax = np.argmin(np.abs(kin.times - kin.t_qmax))
S_at_qmax = kin.means[i_qmax]
t_tang = np.linspace(kin.t_lag - 2, kin.t_qmax + 5, 80)
S_tang = S_at_qmax + slope_val * (t_tang - kin.t_qmax)
ax.plot(t_tang, S_tang, '--', color='red', linewidth=2, alpha=0.7)

# t_lag
ax.axvline(kin.t_lag, color='purple', linestyle='-.', alpha=0.5, linewidth=1.5)
ax.plot(kin.t_lag, kin.S0, 'D', color='purple', markersize=10, zorder=5)
ax.text(kin.t_lag, -1.5, f'$t_{{lag}}$\n{kin.t_lag:.1f}h', ha='center', fontsize=10,
        color='purple', fontweight='bold')

# q_max star
ax.plot(kin.t_qmax, S_at_qmax, '*', color='red', markersize=18, zorder=5)
ax.annotate(f'$q_{{max}}$ = {kin.q_max:.2f}\n@ {kin.t_qmax:.0f}h',
            xy=(kin.t_qmax, S_at_qmax), xytext=(kin.t_qmax + 8, S_at_qmax + 2),
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# t_50
if kin.t_50:
    S50 = kin.S0 - 0.5 * kin.delta_S
    ax.plot(kin.t_50, S50, 's', color='#9C27B0', markersize=9, zorder=5)
    ax.text(kin.t_50, -1.5, f'$t_{{50}}$\n{kin.t_50:.1f}h', ha='center', fontsize=10,
            color='#9C27B0', fontweight='bold')
    ax.axhline(S50, color='#9C27B0', linestyle=':', alpha=0.3)

# t_90
if kin.t_90:
    S90 = kin.S0 - 0.9 * kin.delta_S
    ax.plot(kin.t_90, S90, 's', color='#E91E63', markersize=9, zorder=5)
    ax.text(kin.t_90, -1.5, f'$t_{{90}}$\n{kin.t_90:.1f}h', ha='center', fontsize=10,
            color='#E91E63', fontweight='bold')

# t_depletion
if kin.t_depletion:
    dep_th = max(0.5, 0.05 * kin.S0)
    ax.axvline(kin.t_depletion, color='red', linestyle='--', alpha=0.4)
    ax.plot(kin.t_depletion, dep_th, 'v', color='red', markersize=12, zorder=5)
    ax.text(kin.t_depletion, -1.5, f'$t_{{dep}}$\n{kin.t_depletion:.1f}h', ha='center',
            fontsize=10, color='red', fontweight='bold')

# Active period bracket
if kin.t_lag and kin.t_depletion:
    y_br = -3.2
    ax.annotate('', xy=(kin.t_lag, y_br), xytext=(kin.t_depletion, y_br),
                arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=2))
    ax.text((kin.t_lag + kin.t_depletion) / 2, y_br - 0.8,
            f'$t_{{active}}$ = {kin.t_active:.1f}h  |  $q_{{active}}$ = {kin.q_active:.2f} g/L/h',
            ha='center', fontsize=10, color='darkgreen', fontweight='bold')

ax.set_xlabel('Time (h)', fontsize=12)
ax.set_ylabel('Glucose (g/L)', fontsize=12)
ax.set_title(f'Glucose — All Parameters at a Glance ({DATASET} experiment)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, kin.S0 * 1.15)
ax.set_xlim(kin.times[0] - 3, kin.times[-1] + 15)
fig.tight_layout()
plt.savefig('../output/demo_8_all_params_annotated.png', dpi=150, bbox_inches='tight')
plt.show()
print('↑ 하나의 기질에 대한 모든 파라미터가 어디서 나오는지 한눈에 보여주는 그래프')

---
## Summary

```
CSV 입력  →  loader.py (자동감지)  →  stats.py (replicate 집계)
                                           ↓
                                    kinetics.py (rate, lag, depletion)
                                           ↓
                                    diauxic.py (교차 기질 분석)
                                           ↓
                              pipeline.py (summary.csv + timeseries.csv)
```

**사용법**: `DATASET` 변수를 `'1:1'` 또는 `'2:1'`로 바꾸고 전체 셀 실행 → 해당 데이터 분석 결과 출력